# 12 · Loop Control & Budget Guards

An agent loop can spin forever or quietly burn money. A **`Budget`** sets hard limits on steps, USD, and tokens; a **`Guard`** checks them against a live `CostTracker` and **raises before** the next expensive call. This is the safety rail every long-running agent needs.

> Runs on Ollama (cost stays $0, but the token/step guards still fire).

In [ ]:
import sys, pathlib
p = pathlib.Path.cwd()
root = next(x for x in [p, *p.parents] if (x / 'agents_lab').exists())
sys.path.insert(0, str(root))

A **step budget** stops a runaway loop. `Guard.tick()` raises `RunawayLoop` once the cap is crossed.

In [ ]:
from agents_lab import Budget, Guard, RunawayLoop

guard = Guard(Budget(max_steps=3))
try:
    for i in range(10):
        guard.tick()
        print('step', i + 1, 'ok; remaining:', guard.remaining_steps())
except RunawayLoop as e:
    print('stopped:', e)

A **cost budget** guards spend. Drive a real agent and enforce a max-USD ceiling against its tracker (use a cloud provider to see it bite; on Ollama cost is $0).

In [ ]:
from agents_lab import Agent, BudgetExceeded

agent = Agent(provider='ollama', max_steps=3)
guard = Guard(Budget(max_steps=5, max_usd=0.01, max_tokens=100000))
res = agent.run('What is 8 * 9?')
try:
    guard.tick(agent.tracker)
    print('within budget:', agent.tracker.summary())
except BudgetExceeded as e:
    print('budget hit:', e)

**Takeaway:** never ship an unbounded agent. Cap steps, cost, and tokens — and check them *before* each call, not after the bill arrives.

Next → `13_agent_memory.ipynb`